## 1. Cleaning Cardiac Complications Table

In [10]:
import pandas as pd
import numpy as np
df = pd.read_csv("cardiac_complications.csv")
print(df.shape)
print(df.head())

(2008, 14)
   inpatient_number  nyha_cardiac_function_classification  killip_grade  \
0            848044                                     3             1   
1            821472                                     3             1   
2            866373                                     3             2   
3            793147                                     3             3   
4            739615                                     3             2   

   myocardial_infarction  congestive_heart_failure  \
0                      0                         0   
1                      0                         1   
2                      0                         1   
3                      0                         0   
4                      0                         0   

   peripheral_vascular_disease type_of_heart_failure  lvef  \
0                            0                  Both   NaN   
1                            0                  Both   NaN   
2                          


## Renaming columns
Raw column names like inpatient_number and nyha_cardiac_function_classification aren't clear or consistent. We rename them so the rest of the notebook (and anyone reading it later) can understand what each column actually means, and so column names match across all our tables.


In [20]:
# Mapping old column names to new column names
df = df.rename(columns={"inpatient_number": "patient_id"})
df = df.rename(columns={"nyha_cardiac_function_classification": "heart_failure_severity"})
df = df.rename(columns={"killip_grade": "heart_damage_level"})
df = df.rename(columns={"myocardial_infarction": "heart_attack_history"})
df = df.rename(columns={"type_of_heart_failure": "heart_failure_type"})
df = df.rename(columns={"ea": "ea_ratio"})
df = df.rename(columns={"tricuspid_valve_return_velocity": "pulmonary_velocity"})
df = df.rename(columns={"tricuspid_valve_return_pressure": "pulmonary_pressure"})
df = df.rename(columns={"lvef": "heart_pumping_efficiency"})
df = df.rename(columns={"left_ventricular_end_diastolic_diameter_lv": "left_ventricle_size"})
print(df.columns)
print(df.dtypes)
df.

Index(['patient_id', 'heart_failure_severity', 'heart_damage_level',
       'heart_attack_history', 'congestive_heart_failure',
       'peripheral_vascular_disease', 'heart_failure_type',
       'heart_pumping_efficiency', 'left_ventricle_size', 'mitral_valve_ems',
       'mitral_valve_ams', 'ea_ratio', 'pulmonary_velocity',
       'pulmonary_pressure'],
      dtype='str')
patient_id                        int64
heart_failure_severity            int64
heart_damage_level                int64
heart_attack_history              int64
congestive_heart_failure          int64
peripheral_vascular_disease       int64
heart_failure_type             category
heart_pumping_efficiency        float64
left_ventricle_size             float64
mitral_valve_ems                float64
mitral_valve_ams                float64
ea_ratio                        float64
pulmonary_velocity              float64
pulmonary_pressure              float64
dtype: object


## Check for data types
We convert heart_failure_type to the category dtype, which is more memory-efficient and semantically correct for columns with a fixed set of repeating values.

In [14]:
df["heart_failure_type"] = df["heart_failure_type"].astype("category")


## Check for duplicate records

Duplicate rows can silently inflate patient counts and bias any analysis or model built on this data. We count exact duplicate rows and inspect them, so we can decide whether they're true duplicates (safe to drop) or coincidentally identical valid records.

In [3]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate patient IDs:", df["patient_id"].duplicated().sum())

Duplicate rows: 0
Duplicate patient IDs: 0


## Missing value Analysis

We check every column for missing values to understand exactly where the data is incomplete. This tells us which columns need a decision — fill with a sensible value, leave as missing because the gap is meaningful, or investigate further — rather than assuming the dataset is complete. We did not impute these missing measurements because it would reduce the number of usable patient records and weaken predictive models. cardiac_complications contains numerous blank entries in these fields.
Removing all rows with missing values would discard a large amount of clinical information and reduce statistical power. It will preserves valuable patient records, improves model stability, reduces loss of information & will increases sample size for machine learning.

In [17]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

print(missing.sort_values("missing_percent", ascending=False))

                             missing_count  missing_percent
pulmonary_pressure                    1826        90.936255
ea_ratio                              1615        80.428287
mitral_valve_ams                      1458        72.609562
heart_pumping_efficiency              1373        68.376494
pulmonary_velocity                    1218        60.657371
mitral_valve_ems                      1028        51.195219
left_ventricle_size                    697        34.711155
congestive_heart_failure                 0         0.000000
peripheral_vascular_disease              0         0.000000
heart_damage_level                       0         0.000000
heart_failure_severity                   0         0.000000
patient_id                               0         0.000000
heart_attack_history                     0         0.000000
heart_failure_type                       0         0.000000


## Checking for Outliers
Potential outliers in mitral_valve_ems and mitral_valve_ams were identified using the IQR method. Extremely large values were further investigated and imputed.

In [23]:
echo_cols = [
    'mitral_valve_ems',
    'mitral_valve_ams',
    'ea_ratio',
    'pulmonary_velocity',
    'pulmonary_pressure'
]

for col in echo_cols:
    print(
        col,
        df[col].describe()
    )

mitral_valve_ems count    980.000000
mean       4.849592
std       38.588059
min        0.030000
25%        0.767500
50%        1.010000
75%        1.280000
max      409.000000
Name: mitral_valve_ems, dtype: float64
mitral_valve_ams count    550.000000
mean       4.086738
std       34.597588
min        0.060000
25%        0.590000
50%        0.830000
75%        1.050000
max      408.000000
Name: mitral_valve_ams, dtype: float64
ea_ratio count    393.000000
mean       1.289686
std        1.289603
min        0.061600
25%        0.656000
50%        0.980000
75%        1.606000
max       21.300000
Name: ea_ratio, dtype: float64
pulmonary_velocity count    790.000000
mean       2.990506
std        0.618153
min        0.900000
25%        2.600000
50%        2.900000
75%        3.347500
max        5.760000
Name: pulmonary_velocity, dtype: float64
pulmonary_pressure count    182.000000
mean      35.910989
std       13.698481
min        1.700000
25%       27.250000
50%       34.000000
75%      

In [25]:
for col in echo_cols:
    
    q1 = df[col].quantile(.25)
    q3 = df[col].quantile(.75)

    iqr = q3-q1

    lower = q1-2.0*iqr
    upper = q3+2.0*iqr

    outliers = df[
        (df[col] < lower)
        |
        (df[col] > upper)
    ]

    print(
        col,
        len(outliers)
    )

mitral_valve_ems 29
mitral_valve_ams 9
ea_ratio 5
pulmonary_velocity 10
pulmonary_pressure 3


In [29]:
df.loc[df["mitral_valve_ems"] > 10, "mitral_valve_ems"] = np.nan
df.loc[df["mitral_valve_ams"] > 10, "mitral_valve_ams"] = np.nan

## New column for severe heart failure flag
High-risk patients were spread across several variables.
Patients with: heart_failure_severity ≥ 3 &  heart_damage_level ≥ 3
represent clinically severe heart failure. Cardiac_complications includes both severity scales. It Improve analysis by faster patient stratification, better risk prediction & for easier dashboard implementation

In [31]:
df[
    'severe_hf'
] = np.where(
    (
        df[
            "heart_failure_severity"
        ] >= 3
    )
    &
    (
        df[
            "heart_damage_level"
        ] >= 3
    ),
    1,
    0
)

## Quality check

In [32]:
df.info()

df.describe(
    include='all'
)

<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   patient_id                   2008 non-null   int64   
 1   heart_failure_severity       2008 non-null   int64   
 2   heart_damage_level           2008 non-null   int64   
 3   heart_attack_history         2008 non-null   int64   
 4   congestive_heart_failure     2008 non-null   int64   
 5   peripheral_vascular_disease  2008 non-null   int64   
 6   heart_failure_type           2008 non-null   category
 7   heart_pumping_efficiency     635 non-null    float64 
 8   left_ventricle_size          1311 non-null   float64 
 9   mitral_valve_ems             969 non-null    float64 
 10  mitral_valve_ams             543 non-null    float64 
 11  ea_ratio                     393 non-null    float64 
 12  pulmonary_velocity           790 non-null    float64 
 13  pulmonary_pres

,patient_id,heart_failure_severity,heart_damage_level,heart_attack_history,congestive_heart_failure,peripheral_vascular_disease,heart_failure_type,heart_pumping_efficiency,left_ventricle_size,mitral_valve_ems,mitral_valve_ams,ea_ratio,pulmonary_velocity,pulmonary_pressure,severe_hf
count,2008.000000,2008.000000,2008.000000,2008.000000,2008.000000,2008.000000,2008,635.000000,1311.000000,969.000000,543.000000,393.000000,790.000000,182.000000,2008.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,Both,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,1480,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,797747.542829,3.130976,1.992530,0.071215,0.932271,0.050299,NaN,50.682992,53.110015,1.060475,0.833713,1.289686,2.990506,35.910989,0.192231
std,41127.801740,0.682383,0.759884,0.257248,0.251343,0.218615,NaN,13.221701,10.924505,0.458945,0.317271,1.289603,0.618153,13.698481,0.394152
min,722128.000000,2.000000,1.000000,0.000000,0.000000,0.000000,NaN,5.000000,0.300000,0.030000,0.060000,0.061600,0.900000,1.700000,0.000000
25%,763164.500000,3.000000,1.000000,0.000000,1.000000,0.000000,NaN,41.000000,45.000000,0.760000,0.580000,0.656000,2.600000,27.250000,0.000000
50%,798758.000000,3.000000,2.000000,0.000000,1.000000,0.000000,NaN,51.000000,53.000000,1.000000,0.820000,0.980000,2.900000,34.000000,0.000000
75%,829399.750000,4.000000,2.000000,0.000000,1.000000,0.000000,NaN,61.000000,60.000000,1.260000,1.045000,1.606000,3.347500,43.000000,0.000000


## Save clean dataset

In [ ]:
#df.to_csv(
    "cardiac_complications_cleaned.csv",
    index=False
)